In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2, SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2 # 16 qubits (for more than 24 we need to install Qiskit air)
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
from qiskit import QuantumRegister, ClassicalRegister, qpy, transpile
from collections import Counter
from qiskit_aer import AerSimulator, Aer
from dataclasses import dataclass
from typing import List, Tuple
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:

def visualize_data(train_data, train_labels, test_data, test_predictions, k_value):
    dims = len(train_data[0])
    is_1d = (dims == 1)

    train_x = [pt[0] for pt in train_data]
    test_x  = [pt[0] for pt in test_data]
    train_y = [0 if is_1d else pt[1] for pt in train_data]
    test_y  = [0 if is_1d else pt[1] for pt in test_data]

    if is_1d:
        plt.figure(figsize=(10, 2))
        plt.axhline(0, color='black', linewidth=1, zorder=1)
        plt.yticks([]); plt.ylim(-0.5, 0.5); plt.grid(False)
        for s in ['left', 'right', 'top']: plt.gca().spines[s].set_visible(False)
    else:
        plt.figure(figsize=(8, 6))
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.ylabel("Dimension 2")

    unique_labels = sorted(list(set(train_labels)))
    cm = plt.get_cmap('tab10')
    color_map = {l: cm(i) for i, l in enumerate(unique_labels)}

    plt.scatter(train_x, train_y, s=100, zorder=2, edgecolors='white',
                c=[color_map[l] for l in train_labels], label="Train")
    
    plt.scatter(test_x, test_y, s=180, zorder=3, marker='x', linewidths=4,
                c=[color_map[l] for l in test_predictions], label='Test (Predicted)')

    for label, color in color_map.items():
        plt.scatter([], [], c=[color], label=label)

    proj_text = " (Projection)" if dims > 2 else ""
    plt.title(f"{dims}D QKNN Feature Space{proj_text} | K={k_value}")
    plt.xlabel("Dimension 1")
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

@dataclass
class ClassicalInput:
    train: List[np.ndarray]
    test: List[np.ndarray]

@dataclass
class QuantumInput:
    train: List[QuantumCircuit]
    test: List[QuantumCircuit]

QKNNInput = ClassicalInput | QuantumInput

def encode_classical_data(data: np.ndarray) -> QuantumCircuit:
    qc = QuantumCircuit(len(data))
    for i, val in enumerate(data):
        qc.ry(val, i)
    return qc

def get_initialization_circuits(qknn_input: QKNNInput, train_i: int, test_i: int) -> Tuple[QuantumCircuit]:
    match qknn_input:
        case QuantumInput(train, test):
            return test[test_i]
        case ClassicalInput(train, test):
            return encode_classical_data(train[train_i]), encode_classical_data(test[test_i])


def compute_fidelity(M, n_index, train_init, test_init, show_circuit, shots, backend, format):
    
    n_qubits = test_init.num_qubits
    
    ancilla = QuantumRegister(1, name="ancilla")
    reg_index = QuantumRegister(n_index, name="index")
    reg_a = QuantumRegister(n_qubits, name="train")
    reg_b = QuantumRegister(n_qubits, name="test")
    qc = QuantumCircuit(ancilla, reg_index, reg_a, reg_b)

    # 1. State Preparation
    qc.append(train_init, list(reg_index) + list(reg_a))
    qc.append(test_init, list(reg_b))
    qc.barrier()

    # 2. SWAP Test
    qc.h(ancilla)
    for i in range(n_qubits):
        qc.cswap(ancilla[0], reg_a[i], reg_b[i])
    qc.h(ancilla)
    
    # 3. Measuring the qubits
    meas_reg_len = n_index + 1
    meas_reg = ClassicalRegister(meas_reg_len, "meas")
    qc.add_register(meas_reg)
    qc.measure(list(reg_index) + list(ancilla), meas_reg)
  

    if show_circuit:
        if format == "mpl":
            display(qc.draw(format))
        if format == "latex_source":
            print(qc.draw(format))

    backend = Aer.get_backend('qasm_simulator')
    job = backend.run(transpile(qc, backend), shots = shots)
    result = job.result()
    counts_knn = result.get_counts()
    
    # Count how many times we measure 0 and how many times we measure 1
    counts_by_index = np.zeros((2**n_index, 2))
    for b, count in counts_knn.items():
        anc_bit = b[0]
        idx_bits = b[1:1+n_index]
        i = int(idx_bits, 2)      # Find the index in counts_by_index to which the bit string idx_bits corresponds

        # Add to corresponding place in array based on measuring 0 or 1
        if int(anc_bit) == 0:
            counts_by_index[i, 0] += count
        else:
            counts_by_index[i, 1] += count
    
    fidelities = []

    for i in range(M):
        counts0, counts1 = counts_by_index[i]
        total = counts0 + counts1

        if total == 0:
            fidelities.append(np.nan)
        else:
            prob0 = counts0/total # Probability of measuring 0
            fidelities.append(2*prob0 -1)

    
    return fidelities

def qknn(M, num_index, qknn_input: QKNNInput, train_labels, test_labels, K=1, backend=FakeGuadalupeV2(), shots = 10000, show_circuit=True, format = "mpl"):        
    predictions = []
    num_test = len(qknn_input.test)
    
    for i in range(num_test):

        test_qc = get_initialization_circuits(qknn_input, 0, i)
        fid = compute_fidelity(M, num_index, qknn_input.train, test_qc, show_circuit, shots, backend, format)
        show_circuit = False 

        for j in range(M):    
            print(f"Fidelity(Test {i}, Train {j} [{train_labels[j]}]): {fid[j]:.4f}")

        sorted_idx = np.argsort(fid)[::1] #Sorting indices
        top_k_labels = train_labels[sorted_idx[:K]]
        prediction = Counter(top_k_labels).most_common(1)[0][0]
        predictions.append(prediction)
        
        status = ""
        if int(prediction) == int(test_labels[i]):
            status = f" ✅"
        else:
            status = f" - Actual: {test_labels[i]} ❌"
        
        print(f"Test {i}: top k nearest neighbors = {top_k_labels} | Prediction: {prediction}{status}")

    correct = sum(1 for p, a in zip(predictions, test_labels) if p == a)
    accuracy = (correct/num_test)*100
    print(f"OVERALL ACCURACY: {accuracy:.2f}%")

    if isinstance(qknn_input, ClassicalInput):
        visualize_data(qknn_input.train, train_labels, qknn_input.test, predictions, K)

    return accuracy, predictions

In [3]:
# Load gate W representing the train states and the list with corresponding entanglement classes
with open("W_gate.qpy", "rb") as handle:

    qc_train = qpy.load(handle)

labels_train = np.load("Class list.npy")   

# Load the V gates representing the test states and the list with corresponding entanglement classes
with open("V_gate list.qpy", "rb") as handle:
    qc_test = qpy.load(handle)

labels_test = np.load("Class list test.npy")

print(labels_train)

[1. 0. 1. 0. 1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 1. 0. 0. 1. 1.
 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 1.
 0. 1. 1. 1. 1. 0. 0. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.
 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0. 0. 0. 1. 1. 0. 0.
 0. 0. 0. 1.]


In [4]:
model_acc, preditions_test = qknn(100, 7,
    qknn_input=QuantumInput(
    train= qc_train[0],
    test = qc_test
    ), K = 10,
    train_labels=labels_train, 
    test_labels=labels_test,
    show_circuit=True, format = "latex_source"
)

\documentclass[border=2px]{standalone}

\usepackage[braket, qm]{qcircuit}
\usepackage{graphicx}

\begin{document}
\scalebox{1.0}{
\Qcircuit @C=1.0em @R=0.2em @!R { \\
	 	\nghost{{ancilla} :  } & \lstick{{ancilla} :  } & \qw \barrier[0em]{13} & \qw & \gate{\mathrm{H}} & \qw & \qw & \qw & \qw & \qw & \qw & \ctrl{8} & \ctrl{9} & \ctrl{10} & \gate{\mathrm{H}} & \meter & \qw & \qw\\
	 	\nghost{{index}_{0} :  } & \lstick{{index}_{0} :  } & \multigate{9}{\mathrm{gate\,W}}_<<<{0} & \qw & \meter & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw\\
	 	\nghost{{index}_{1} :  } & \lstick{{index}_{1} :  } & \ghost{\mathrm{gate\,W}}_<<<{1} & \qw & \qw & \meter & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw\\
	 	\nghost{{index}_{2} :  } & \lstick{{index}_{2} :  } & \ghost{\mathrm{gate\,W}}_<<<{2} & \qw & \qw & \qw & \meter & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw\\
	 	\nghost{{index}_{3} :  } & \lstick{{index}_{3} :  } & \gh

In [5]:
# # Possible evaluation measures
# def cm(y_test, y_pred_test):
#    """ Visualizes the confusion matrix, which summarizes the performance of "
#        our qknn.
       
#        Parameters:
#           y_test: classes of the test dataset 
#           y_pred_test: classes predicted by the model for the test dataset
      
#        Returns:
#           visualization of the confusion matrix
#    """

 
#    # Finding the confusion matrix
#    cm = confusion_matrix(y_test, y_pred_test)

#    # Defining labels
#    col_labels = []
#    idx_labels = []

#    for i in range(len(np.unique(y_test))):
#       col_labels.append(f"Expected {i}")
#       idx_labels.append(f"Predict {i}")

#    # Visualizing the confusion matrix
#    plt.figure(figsize = (6,4))

#    cm_matrix = pd.DataFrame(data = cm, columns = col_labels, index = idx_labels)

#    sns.heatmap(cm_matrix, annot = True, fmt = 'd', cmap = "crest")

# # Confusion matrix for our data
# cm(labels_test, predictions_test)

# # Classification report
# print(classification_report(labels_test, predictions_test))
# print("""
# Here the precision is the percentage of correctly predicted positive outcomes
# out of all the predicted positive outcomes. The recall or sensitivity is the 
# ratio of percentage of correcty predicted positive outcomes out of all the  
# positive outcomes. Recall is the weighted harmonic mean of the precision and 
# recall.""")


# # Testing for overfitting 

# def overfit(y_test, y_train, y_pred_test, y_pred_train, p):
#     score_train = accuracy_score(y_train, y_pred_train)
#     score_test = accuracy_score(y_test, y_pred_test)

#     diff = np.abs(score_train - score_test)

#     if diff <= p:
#         print("There is no question of overfitting.")
    
#     else:
#         print("There is a question of overfitting.")
